# Feature Engineering

Feature engineering transforms the raw serve-level observations into a richer representation that machine learning models can exploit more effectively. Raw columns such as `serve_type` or `spin_intensity` encode serve attributes independently, but the predictive value of a serve is rarely attributable to any single attribute in isolation. This notebook constructs features that capture game-state pressure, tactical interactions among serve attributes, opponent characteristics, and the historical performance of specific serve combinations.

All features constructed here are restricted to information that is available before the serve is executed. This constraint is essential: the eventual model must be deployable as a real-time decision-support tool, so it cannot use any information that depends on the serve outcome or subsequent rally dynamics.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/table_tennis_serves.csv")

df.head()

## Data Validation

Before engineering any features, we inspect the raw dataset to confirm that it loaded correctly and that the outcome column has a plausible distribution. Verifying the row count, column set, and class balance at this stage prevents downstream errors that would otherwise be difficult to trace back to a loading problem. An unexpected class imbalance in `point_outcome` would also influence the modeling strategy in the next notebook.

In [ ]:
original_shape = df.shape
print("Dataset shape:", original_shape)
print()
print("Value counts for point_outcome:")
print(df["point_outcome"].value_counts())

## Target Variable Construction

The modeling objective is binary classification: predicting whether the server wins the point. We encode this as `point_won`, where a value of 1 indicates that the server won the point and a value of 0 indicates that the server lost it. Binary encoding is preferred over using the raw string `point_outcome` column because it is directly compatible with standard classification algorithms and evaluation metrics such as ROC-AUC.

In [ ]:
df["point_won"] = (df["point_outcome"] == "won").astype(int)

## Data Leakage Identification

Data leakage occurs when information that would not be available at prediction time is included in the model's feature set. In this dataset, several columns describe what happened after the serve was executed: how the opponent returned the ball, how long the rally lasted, and how the point ended. Including these variables would produce artificially optimistic evaluation metrics during training, but the model would fail completely in deployment because none of these values can be known before the serve. We identify and exclude them explicitly to prevent this failure mode.

In [ ]:
leakage_features = [
    "return_type",
    "return_quality",
    "return_placement",
    "rally_length",
    "point_end_type",
    "rally_type_achieved",
    "chop_rally_outcome"
]
leakage_features

## Game State Features

The score at the time of the serve carries substantial tactical meaning. A server who is tied at 10-10 faces different psychological and strategic pressures than one who leads 8-4. We construct a set of binary flags and a continuous margin variable to capture these game-state conditions. Each flag has a specific tactical interpretation: `is_tied` identifies a neutral high-stakes moment, `is_trailing` and `is_leading` encode the direction of the score differential, `is_late_game` flags the final stretch of a standard game, and `is_deuce_or_later`, `is_game_point_for_server`, and `is_game_point_against_server` encode the highest-pressure moments where a single point can determine who wins the game.

In [ ]:
df["score_margin"] = df["server_score"] - df["receiver_score"]
df["total_points_played_in_game"] = df["server_score"] + df["receiver_score"]
df["is_tied"] = (df["server_score"] == df["receiver_score"]).astype(int)
df["is_trailing"] = (df["server_score"] < df["receiver_score"]).astype(int)
df["is_leading"] = (df["server_score"] > df["receiver_score"]).astype(int)
df["is_late_game"] = (df["total_points_played_in_game"] >= 16).astype(int)
df["is_deuce_or_later"] = ((df["server_score"] >= 10) & (df["receiver_score"] >= 10)).astype(int)
df["is_game_point_for_server"] = ((df["server_score"] >= 10) & (df["server_score"] > df["receiver_score"])).astype(int)
df["is_game_point_against_server"] = ((df["receiver_score"] >= 10) & (df["receiver_score"] > df["server_score"])).astype(int)

## Interaction Features

Game-state flags and serve attributes are informative on their own, but certain combinations carry signal that neither variable encodes alone. We construct three interaction features to capture these joint effects. The `spin_x_looper` feature tests whether a high-spin serve directed against a looping opponent is more effective than either factor would predict independently, since looping opponents are specifically sensitive to incoming spin. The `score_margin_abs` feature measures how far the score is from a tie in a direction-agnostic way, which is useful for capturing pressure that arises both when the server is well ahead and when the server is well behind. The `is_high_pressure` flag collapses three correlated binary flags into a single indicator for any situation in which a game-deciding point could occur, reducing multicollinearity without sacrificing information.

In [ ]:
# Opponent style dummies are created here because spin_x_looper depends on opponent_is_looper
df["opponent_is_looper"] = (df["opponent_style"] == "looper").astype(int)
df["opponent_is_chopper"] = (df["opponent_style"] == "chopper").astype(int)
df["opponent_is_attacker"] = (df["opponent_style"] == "attacker").astype(int)

# Spin intensity amplified against looping opponents
df["spin_x_looper"] = df["spin_intensity"] * df["opponent_is_looper"]

# Pressure distance: how far from a tie, regardless of direction
df["score_margin_abs"] = df["score_margin"].abs()

# Unified high-pressure flag combining three correlated game-state conditions
df["is_high_pressure"] = (
    (df["is_deuce_or_later"] == 1) |
    (df["is_game_point_for_server"] == 1) |
    (df["is_game_point_against_server"] == 1)
).astype(int)

print("Interaction features added.")
print(df[["spin_x_looper", "score_margin_abs", "is_high_pressure"]].describe())

## Opponent Encoding

The `opponent_skill_level` column contains categories that have a natural order: a beginner opponent is meaningfully less challenging than an intermediate opponent, and so on through advanced and expert. Ordinal encoding maps these levels to integers 1 through 4, preserving that order so that models can learn a monotonic relationship between opponent difficulty and serve effectiveness. A nominal encoding (such as one-hot encoding) would discard this ordering information entirely, which would be a loss of domain knowledge.

In [ ]:
skill_map = {"beginner": 1, "intermediate": 2, "advanced": 3, "expert": 4}
df["opponent_skill_numeric"] = df["opponent_skill_level"].map(skill_map).fillna(2)

print("Value counts for opponent_skill_numeric:")
print(df["opponent_skill_numeric"].value_counts().sort_index())

## Serve Combination Features

A serve is not a single choice but a combination of simultaneously selected attributes: serve type, spin type, length, and placement zone. The tactical effect of, for example, a short backspin serve to the forehand corner differs from a short backspin serve to the body in ways that cannot be captured by modeling each attribute independently. We construct string concatenations of attribute pairs and of all four attributes together to create explicit combination identifiers. These string features are treated as categorical variables in the modeling step, so that models can learn serve-combination-level patterns rather than only attribute-level patterns.

In [ ]:
df["serve_spin_combo"] = df["serve_type"] + "_" + df["spin_type"]
df["serve_length_spin_combo"] = df["serve_length"] + "_" + df["spin_type"]
df["serve_placement_combo"] = df["serve_type"] + "_" + df["placement_zone"]
df["full_serve_combo"] = (
    df["serve_type"] + "_" +
    df["spin_type"] + "_" +
    df["serve_length"] + "_" +
    df["placement_zone"]
)

## Spin Intensity Flags

Spin intensity is measured on a continuous numeric scale, but its tactical implications are often more naturally described in discrete terms. A heavy-spin serve requires fundamentally different adjustments from the receiver than a no-spin serve, and the difference between intensities 3 and 4 is categorically distinct from the difference between 1 and 2. Discretizing spin intensity into binary flags for heavy spin (intensity at or above 3) and low spin (intensity at or below 1) allows models to detect these threshold effects directly. The `spin_length_interaction` string feature combines the discrete spin level with serve length to identify tactics like a heavy-spin short serve, which has a specific and well-known tactical advantage.

In [ ]:
df["is_heavy_spin"] = (df["spin_intensity"] >= 3).astype(int)
df["is_low_spin"] = (df["spin_intensity"] <= 1).astype(int)
df["spin_length_interaction"] = df["spin_intensity"].astype(str) + "_" + df["serve_length"]

## Historical Combo Statistics

Knowing that a particular serve combination has a high historical win rate is valuable predictive information, but a raw win rate computed from a small sample is unreliable. A serve combination attempted only twice could show a 100% win rate purely by chance. We address this with two complementary features. First, we compute `combo_win_rate` as the observed fraction of won points across all historical uses of each serve combination. Second, we compute `combo_reliability` as the fraction of a 30-attempt reference sample that has been observed, capped at 1.0. This reliability score allows the model and the recommendation system to discount win rates that are based on few observations while fully trusting those based on 30 or more.

In [ ]:
combo_summary = (
    df.groupby("full_serve_combo")
    .agg(
        combo_attempts=("point_won", "count"),
        combo_win_rate=("point_won", "mean")
    )
    .reset_index()
)
df = df.merge(combo_summary, on="full_serve_combo", how="left")
df["combo_reliability"] = np.minimum(df["combo_attempts"] / 30, 1)

## Feature Validation

Three sanity checks are applied before saving the engineered dataset. First, we compare the column count against the original to confirm that all features were appended as expected. A discrepancy would indicate a naming collision or a silent failure in a merge. Second, we check for null values in the engineered columns, since any null in a feature used by the model will cause a runtime error during inference. Third, we compute the absolute Pearson correlation of each new numeric feature with the target variable and display the top 10, which serves as a preliminary signal of feature relevance and helps identify any features that have unexpectedly high correlation that might indicate leakage.

In [ ]:
# Check 1: Column count comparison
engineered_count = df.shape[1] - original_shape[1]
print(f"Original column count : {original_shape[1]}")
print(f"Current column count  : {df.shape[1]}")
print(f"Engineered features   : {engineered_count}")
print()

# Check 2: Null check on new columns only
new_columns = df.columns[original_shape[1]:].tolist()
null_counts = df[new_columns].isnull().sum()
print("Null values in engineered columns:")
print(null_counts[null_counts > 0] if null_counts.any() else "  None — all engineered columns are complete.")
print()

# Check 3: Top-10 numeric features by absolute correlation with point_won
numeric_new = df[new_columns].select_dtypes(include=[np.number]).columns.tolist()
correlations = (
    df[numeric_new + ["point_won"]]
    .corr()["point_won"]
    .drop("point_won", errors="ignore")
    .abs()
    .sort_values(ascending=False)
    .head(10)
)
print("Top-10 engineered features by |correlation| with point_won:")
print(correlations.to_string())

## Feature Summary

The table below consolidates all engineered features by category, together with their data types and Pearson correlations with the target variable. This summary provides a structured reference for the feature set that will be passed to the modeling notebook. Features marked as categorical (dtype `object`) will be handled by the one-hot encoding step in the preprocessing pipeline.

In [ ]:
feature_groups = {
    "Game State": [
        "score_margin", "total_points_played_in_game", "is_tied", "is_trailing",
        "is_leading", "is_late_game", "is_deuce_or_later",
        "is_game_point_for_server", "is_game_point_against_server"
    ],
    "Interaction": ["spin_x_looper", "score_margin_abs", "is_high_pressure"],
    "Serve Combinations": [
        "serve_spin_combo", "serve_length_spin_combo",
        "serve_placement_combo", "full_serve_combo"
    ],
    "Spin Flags": ["is_heavy_spin", "is_low_spin", "spin_length_interaction"],
    "Opponent": [
        "opponent_is_looper", "opponent_is_chopper",
        "opponent_is_attacker", "opponent_skill_numeric"
    ],
    "Combo Statistics": ["combo_attempts", "combo_win_rate", "combo_reliability"]
}

rows = []
for group, features in feature_groups.items():
    for feature in features:
        if feature not in df.columns:
            continue
        dtype = str(df[feature].dtype)
        if pd.api.types.is_numeric_dtype(df[feature]):
            win_rate_corr = round(df[feature].corr(df["point_won"]), 4)
        else:
            win_rate_corr = "N/A"
        rows.append({"group": group, "feature": feature, "dtype": dtype, "win_rate_corr": win_rate_corr})

summary_df = pd.DataFrame(rows, columns=["group", "feature", "dtype", "win_rate_corr"])
print(summary_df.groupby("group").apply(lambda x: x[["feature", "dtype", "win_rate_corr"]].to_string(index=False)).to_string())

## Save and Verify

The fully engineered dataset is written to the `data/processed/` directory so that the modeling notebook can load it directly. We then reload the file and confirm that the row and column counts match the in-memory dataframe, which guards against silent truncation during the CSV write operation.

In [ ]:
output_path = "../data/processed/table_tennis_serves_features.csv"
df.to_csv(output_path, index=False)

In [ ]:
import os

saved_df = pd.read_csv(output_path)
print("File saved successfully.")
print(f"  Path  : {os.path.abspath(output_path)}")
print(f"  Shape : {saved_df.shape[0]} rows x {saved_df.shape[1]} columns")

The processed dataset is now ready for consumption by the modeling notebook. All features included are pre-serve variables, ensuring that no outcome information has been allowed to influence the predictor space.

## Feature Engineering Quality Assurance

As a final programmatic check, we assert that all required engineered columns are present and that none of them contain null values. These assertions serve as a contract between this notebook and the modeling notebook: if any assertion fails, execution halts immediately with a descriptive error message rather than propagating a silent defect into the model training pipeline.

In [ ]:
required_engineered = [
    'score_margin', 'total_points_played_in_game', 'is_tied', 'is_trailing', 'is_leading',
    'serve_spin_combo', 'serve_length_spin_combo', 'serve_placement_combo', 'full_serve_combo',
    'combo_attempts', 'combo_win_rate', 'combo_reliability', 'point_won'
]
missing = [c for c in required_engineered if c not in df.columns]
assert not missing, f'Missing engineered columns: {missing}'
assert df[required_engineered].isnull().sum().sum() == 0, 'Nulls found in required engineered columns'
print('Quality assurance checks passed: all required engineered features exist and contain no nulls.')